## Part 3: Features engineering, split and preprocessing (P3)

### Section 3.1: Setup

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base     = clean["model_base"]   # pre-race columns + target 'podium'
outcomes = clean["outcomes"]     # post-race columns NEVER use as features
tables   = clean["tables"]       # all other cleaned tables

base.shape, outcomes.shape, list(tables.keys())

### Section 3.2: Base table sanity check

The dataset has been cleaned with 0 duplicated rows for (raceId, driverId). The missing values are present in the qualifying-related columns q1_s, q2_S, q3_s, qual_position.

In [ ]:
import pandas as pd

# Missingness across the full base table (all columns, all years)
miss = base.isna().mean().sort_values(ascending=False)

summary = pd.DataFrame({
    "missing_count": base.isna().sum(),
    "missing_share": miss,
    "dtype": base.dtypes,
})
summary["missing_share"] = summary["missing_share"].round(4)

print("=" * 65)
print(f"base shape: {base.shape[0]:,} rows × {base.shape[1]} columns")
print(f"Columns with ANY missing:  {(summary['missing_count'] > 0).sum()}")
print(f"Columns with NO  missing:  {(summary['missing_count'] == 0).sum()}")
print("=" * 65)

print("\nColumns with missing values (sorted):")
print(summary[summary["missing_count"] > 0].to_string())

### Section 3.3: Training start year

The ml table uses qualifying features (qual_position, q1_s, q2_s, q3_s,
gap_to_pole_pct). 

Qualifying data does not exist for most of F1 history (e.g. 1950-1993),
so we need to choose the earliest stable year to include in the ML table.

We look at qualifying coverage per season and pick the earliest year where it's stable.

In [ ]:
# Calendar table needed to bring year/round into standings and results
races = tables["races"][["raceId", "year", "round", "circuitId", "date"]].copy()
races.head()

In [ ]:
# Find the first year with any qualifying data and the first year with high coverage
q_cov_exact = base.groupby("year")["has_qualifying"].mean()

# Years before this have no qualifying data at all
years_with_any = q_cov_exact[q_cov_exact > 0].index.min()
# Years before this have full or near-full coverage
years_with_full = q_cov_exact[q_cov_exact >= 0.90].index.min()

print("=" * 60)
print("First year with ANY qualifying data (has_qualifying > 0):", years_with_any)
print("First year with >=90% coverage:                      ", years_with_full)
print("=" * 60)

print("\nCoverage 1990–1997 (the transition years):")
print(q_cov_exact.loc[1990:1997].round(4))

In [ ]:
# Qualifying coverage per season: share of rows with a q1_s time
cov = base.groupby("year")["has_qualifying"].mean()

# Also look at q1_s, q2_s, q3_s separately
q_cov = base.groupby("year")[["q1_s", "q2_s", "q3_s"]].apply(lambda d: d.notna().mean())

print("has_qualifying per year (first 10 and last 10):")
print(cov.head(10))
print(cov.tail(10))

print("\nq1_s / q2_s / q3_s coverage for 1994-2010:")
print(q_cov.loc[1993:2007])

In [ ]:
print("\nq1_s / q2_s / q3_s coverage for 1994-2010:")
print(q_cov.loc[2006:2024])

**We can see that after 2006, the qualification data is stable and available for q1, q2, and q3. This is due to the fact that the knockout era started after 2006.**

In [ ]:
START_YEAR = 2006
model = base[base["year"] >= START_YEAR].copy()
model = model.sort_values(["year", "round", "raceId", "driverId"]).reset_index(drop=True)

print("Rows kept:", model.shape[0])
print("Years:", model["year"].min(), "-", model["year"].max())
print("Rows dropped:", base.shape[0] - model.shape[0])
print("\nmodel shape:", model.shape)
print("model years:", model["year"].min(), "-", model["year"].max())

In [ ]:
import matplotlib.pyplot as plt

# Coverage per season for each qualifying time column
q_cov = (
    base.groupby("year")[["q1_s", "q2_s", "q3_s"]]
        .apply(lambda d: d.notna().mean())
)

# Focus on the era where qualifying data actually exists
q_cov = q_cov.loc[1994:]

fig, ax = plt.subplots(figsize=(12, 5))

for col, color in zip(["q1_s", "q2_s", "q3_s"],
                      ["tab:blue", "tab:orange", "tab:green"]):
    ax.plot(q_cov.index, q_cov[col], marker="o", label=col, color=color)

ax.axvline(2006, color="red", linestyle="--", label="START_YEAR = 2006")

ax.set_title("Qualifying time coverage per season, 1994–2024")
ax.set_xlabel("Year")
ax.set_ylabel("Share of rows with a time")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../reports/fig_p3_qual_coverage_per_year.png",
            dpi=120, bbox_inches="tight")
plt.show()

**Why we start in 2006**

The plot shows qualifying coverage per season. 1994–95 have near-full q1 coverage, 1996–2002 are patchy, 2003–2005 have q1 only, and starting from 2006 (knockout format) all three (q1, q2, q3) are stable, which is why 2006 is our start year.

| Column | Range 2006–2024 | Meaning |
|---|---|---|
| `q1_s` | 0.97 – 1.00 | Almost every entrant sets a Q1 time |
| `q2_s` | 0.70 – 0.75 | ~15 of ~20 drivers reach Q2 |
| `q3_s` | 0.37 – 0.50 | ~10 of ~20 drivers reach Q3 |

The lower plateaus are structural, not missing data: only the top 15 reach Q2 and the top 10 reach Q3. 

### Section 3.4: Time split

We split by year. The model's job is to predict
future races from pre-race information, so training on later rows and
testing on earlier ones would leak future information into the model. A
time split matches how the model will actually be used.

- **Train**: year ≤ 2019 (5,742 rows)
- **Validation**: 2020–2021 (774 rows)
- **Test**: year ≥ 2022 (1,351 rows)

**Why these boundaries.**
Training covers complete pre-COVID seasons under the V8 and hybrid rules. Validation covers the nearest seasons to the test set and is used for model, threshold and preprocessing choices. Its weakness is that 2020 was shortened by COVID, so we report results per season. Test covers the ground-effect era, a regulation change the model never sees in training, and is used once.

**Class balance.** Train 14.0%, validation 15.1%, test 15.1%, so base rates are similar across splits.

**Alternatives.** A random split leaks same-season information. Random assignment of seasons mixes regulations.

In [ ]:
import numpy as np

model["split"] = np.select(
    [model["year"] <= 2019,
     model["year"].between(2020, 2021),
     model["year"] >= 2022],
    ["train", "validation", "test"],
    default="unknown"
)

print(model["split"].value_counts())
print("\nmodel shape:", model.shape)

In [ ]:
# Class balance per split
balance = model.groupby("split")["podium"].agg(["mean", "count"])
print("Class balance per split:")
print(balance.round(4))

# Plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
balance_ordered = balance.reindex(["train", "validation", "test"])
bars = ax.bar(balance_ordered.index, balance_ordered["mean"] * 100,
              color=["tab:blue", "tab:orange", "tab:green"])
ax.bar_label(bars, fmt="%.1f%%")
ax.set_title("Podium rate per split (class balance)")
ax.set_xlabel("Split")
ax.set_ylabel("Podium rate (%)")
ax.set_ylim(0, 20)
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("../reports/fig_p3_split_balance.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
model.to_csv("../reports/p3_modeling_table.csv", index=False)
print("saved with split, shape:", model.shape)
print("columns:", len(model.columns))

### Section 3.5: Grid and qualifying features

The prediction cutoff is after qualifying and before the race. Every feature
below is set during or before grid formation; none comes from the race itself.

**Grid features**

- `grid_fixed` is the ordinal starting position. The raw `grid` column uses
  `0` for a pit-lane start, which a linear model would read as ahead of pole. We set those to PIT_GRID = 25, one place past the largest grid in the training years (24, from 2010–2012). The value is derived from training data only, and a range check confirms no validation or test grid exceeds the training maximum.
- pit_lane_flag is 1 for pit-lane starts and 0 otherwise. It keeps the start type that grid_fixed compresses into one ordinal value.

**Qualifying features**

- `qual_position` is the driver's rank in the qualifying session.
- `best_qual_s` is the driver's best qualifying lap in seconds
  (`min(q1_s, q2_s, q3_s)` across the sessions they reached).
- `gap_to_pole_pct` is the percentage by which the driver’s best time is slower than the fastest time in the same race. Pole is defined by fastest time, not by grid position, because 61 races have a pole starter who was not the fastest qualifier.
- `no_qual_time` is 1 when the driver set no qualifying time at all
  (`best_qual_s` is NaN). 96 rows (1.2%) fall into this group; 21 of them
  also have no `qual_position`. They are excluded from qualifying,
  withdrawn, or unrecorded, and they start from the back or the pit lane.

  **Why `q1_s`, `q2_s`, `q3_s` are not features.** The raw session times
are used to build `best_qual_s` (the minimum across sessions the driver
reached) and `gap_to_pole_pct` (the normalised gap to the fastest lap in
the race). The raw columns are not in `FEATURES` for three reasons:

1. Their missingness is structural: a Q1-eliminated driver has NaN Q2 and
   Q3. `best_qual_s` and `gap_to_pole_pct` handle that by falling back on
   whichever sessions the driver reached.
2. Absolute lap times vary by track, so the
   raw seconds are not comparable across races. `gap_to_pole_pct`
   normalises by the race's fastest time.
3. Including all three would add three highly correlated columns with
   different missingness patterns for the same underlying fact.

In [ ]:
# Grid features: grid_fixed (ordinal) + pit_lane_flag (indicator)

# 1. Max real grid slot, from TRAIN ONLY.
max_grid = model.loc[
    (model["split"] == "train") & (model["grid"] > 0), "grid"
].max()
print("Max real grid slot (train years only):", max_grid)

# 2. Sentinel: one past the largest real grid slot.
PIT_GRID = max_grid + 1
print(f"PIT_GRID = {PIT_GRID}")

# 3. Build the two features.
model["pit_lane_flag"] = (model["grid"] == 0).astype(int)
model["grid_fixed"] = model["grid"].where(model["grid"] > 0, PIT_GRID)

# 4. Crosstab: pit_lane_flag agrees with pitlane_start.
print("\nCrosstab (grid==0 vs pitlane_start):")
print(pd.crosstab(model["grid"] == 0, model["pitlane_start"]))

# 5. Distribution tail.
print("\nGrid value counts (last 5):")
print(model["grid_fixed"].value_counts().sort_index().tail())

# 6. Assertions.
assert model["grid_fixed"].notna().all(), "null grid after recode"
assert model["grid_fixed"].between(1, PIT_GRID).all()
assert (model.loc[model["pit_lane_flag"] == 1, "grid_fixed"] == PIT_GRID).all()

# 7. Leakage: no val/test grid exceeds the train max.
leak_check = model.loc[
    (model["split"] != "train") & (model["grid"] > max_grid)
]
assert len(leak_check) == 0, \
    f"val/test rows with real grid > train max_grid: {len(leak_check)}"

print(f"\nPit lane starts: {model['pit_lane_flag'].sum()} rows")
print(f"Rows at sentinel ({PIT_GRID}): {(model['grid_fixed'] == PIT_GRID).sum()}")
print(f"Val/test rows with grid above train max ({max_grid}): {len(leak_check)}")
print("Grid features: OK")

In [ ]:
# Qualifying features
model["best_qual_s"] = model[["q1_s", "q2_s", "q3_s"]].min(axis=1)

# Gap to pole (percentage)
pole = model.groupby("raceId")["best_qual_s"].transform("min")
model["gap_to_pole_pct"] = (model["best_qual_s"] - pole) / pole * 100

# No-time flag: 1 when the driver set no qualifying time at all
model["no_qual_time"] = model["gap_to_pole_pct"].isna().astype(int)

print(model[[
    "grid_fixed", "pit_lane_flag", "qual_position",
    "best_qual_s", "gap_to_pole_pct", "no_qual_time",
]].describe())

In [ ]:
defined = model["gap_to_pole_pct"].notna()
assert (model.loc[defined, "gap_to_pole_pct"] >= 0).all(), \
    "negative gap found on rows with a defined gap"

# Pole = fastest best_qual_s in the race, not qual_position == 1
model["is_pole_time"] = (
    model["best_qual_s"]
    == model.groupby("raceId")["best_qual_s"].transform("min")
)
assert (model.loc[model["is_pole_time"] & defined, "gap_to_pole_pct"] == 0).all(), \
    "driver with race-minimum best_qual_s does not have gap 0"

print(f"Rows with defined gap: {defined.sum()}")
print(f"Rows with NaN gap (no qualifying time at all): {(~defined).sum()}")
print(f"Rows with pole-time: {model['is_pole_time'].sum()}")

# Optional: how often qual_position == 1 is not the fastest time
mismatch = model[(model["qual_position"] == 1) & ~model["is_pole_time"]]
print(f"Races where qual_position == 1 is not the fastest qualifier: "
      f"{mismatch['raceId'].nunique()}")

# Confirm the NaN gap is fully explained by missing qualifying data
all_q_missing = model[["q1_s", "q2_s", "q3_s"]].isna().all(axis=1)
assert (all_q_missing == model["gap_to_pole_pct"].isna()).all(), \
    "NaN gap does not match all-three-missing"
print("NaN gap == all three qualifying times missing: OK")

In [ ]:
import numpy as np

model["qual_group"] = np.select(
    [model["q3_s"].notna(),
     model["q2_s"].notna(),
     model["q1_s"].notna()],
    ["Q3", "Q2 only", "Q1 only"],
    default="no qualifying time",
)

print(
    model.groupby("qual_group", observed=True)
         .agg(
             n_rows=("gap_to_pole_pct", "size"),
             n_defined=("gap_to_pole_pct", "count"),
             mean=("gap_to_pole_pct", "mean"),
             median=("gap_to_pole_pct", "median"),
             max=("gap_to_pole_pct", "max"),
         )
         .round(3)
)

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
used = {"grid_fixed", "pit_lane_flag",
        "qual_position", "best_qual_s", "gap_to_pole_pct",
        "no_qual_time"}
assert not (used & FORBIDDEN), "post-race column used as feature"
print("No grid + qualifying leakage check: OK")

### Section 3.6: Pre-race standings

The standings tables record cumulative points and rank **after** each
race. Using them directly would leak the current race's result into the
features.

Fix: sort the standings by `(year, round)`, then `shift(1)` within
`(year, driverId)` and `(year, constructorId)` so each row holds the
standings going into that race. The shift never uses the current race.

Round 1 has no previous race: points are 0 (the true value), position is
undefined (NaN), flagged by `standings_missing`. Coverage gaps — the 49
`(raceId, driverId)` pairs inside the modeling window where the driver
raced but has no standings row — are also filled with 0 for points. A
diagnostic confirmed all 49 are the driver's first standings appearance
of the season, so 0 is the correct value, not a lost earlier total.

We assert: one standings row per `(year, id, round)`, round-1 points
are 0, round-1 position is NaN.

In [ ]:
ds = tables["driver_standings"][
    ["raceId", "driverId", "points", "position"]
].copy()
ds = ds.merge(races, on="raceId", how="left")
ds = ds.sort_values(["year", "round", "driverId"])

assert not ds.duplicated(["year", "driverId", "round"]).any(), \
    "duplicate driver standings row"

# Shift within (year, driverId): round N gets round N-1's values.
ds["pts_before"] = ds.groupby(["year", "driverId"])["points"].shift(1)
ds["pos_before"] = ds.groupby(["year", "driverId"])["position"].shift(1)

# Round 1 has no previous race: points are 0, position is undefined.
ds["pts_before"] = ds["pts_before"].fillna(0.0)
ds["standings_missing"] = ds["pos_before"].isna().astype("int8")

ds_pre = ds[["raceId", "driverId", "pts_before", "pos_before",
             "standings_missing"]]

print("ds_pre shape:", ds_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["pts_before", "pos_before", "standings_missing",
                      "pts_before_x", "pts_before_y",
                      "pos_before_x", "pos_before_y",
                      "standings_missing_x", "standings_missing_y"]],
    errors="ignore",
)

model = model.merge(ds_pre, on=["raceId", "driverId"], how="left")
model["pts_before"] = model["pts_before"].fillna(0.0)

first_race = model.groupby("year")["round"].transform("min") == model["round"]
assert model.loc[first_race, "pts_before"].eq(0.0).all(), "round 1 should have 0 points"
assert model.loc[first_race, "pos_before"].isna().all(), "round 1 position should be NaN"
print("Driver standings: OK")

In [ ]:
cs = tables["constructor_standings"][
    ["raceId", "constructorId", "points", "position"]
].copy()
cs = cs.merge(races, on="raceId", how="left")
cs = cs.sort_values(["year", "round", "constructorId"])

assert not cs.duplicated(["year", "constructorId", "round"]).any(), \
    "duplicate constructor standings row"

cs["cons_pts_before"] = cs.groupby(["year", "constructorId"])["points"].shift(1)
cs["cons_pos_before"] = cs.groupby(["year", "constructorId"])["position"].shift(1)

cs["cons_pts_before"] = cs["cons_pts_before"].fillna(0.0)
cs["cons_standings_missing"] = cs["cons_pos_before"].isna().astype("int8")

cs_pre = cs[["raceId", "constructorId",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]]

print("cs_pre shape:", cs_pre.shape)

In [ ]:
# Safe to rerun: drop existing standings columns first.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["cons_pts_before", "cons_pos_before",
                      "cons_standings_missing",
                      "cons_pts_before_x", "cons_pts_before_y",
                      "cons_pos_before_x", "cons_pos_before_y",
                      "cons_standings_missing_x", "cons_standings_missing_y"]],
    errors="ignore",
)

model = model.merge(cs_pre, on=["raceId", "constructorId"],
                    how="left", validate="many_to_one")
model["cons_pts_before"] = model["cons_pts_before"].fillna(0.0)

assert model.loc[first_race, "cons_pts_before"].eq(0.0).all(), \
    "round 1 should have 0 constructor points"
assert model.loc[first_race, "cons_pos_before"].isna().all(), \
    "round 1 constructor position should be NaN"
print("Constructor standings: OK")

print("model shape:", model.shape)
print(model[["pts_before", "pos_before", "standings_missing",
             "cons_pts_before", "cons_pos_before",
             "cons_standings_missing"]].isna().mean())

### Section 3.7: Circuit history + home flag + era ###

We also need to add these to our features: 

**Circuit history.** Some drivers are specialists at certain tracks (e.g. Senna
at Monaco, Hamilton at Silverstone). We capture this with two features per
(driver, circuit):
- `circuit_starts_before`: number of previous races by this driver at this circuit
- `circuit_podiums_before`: number of previous podiums by this driver at this circuit

Both use only past races. The current race is excluded by `shift(1).cumsum()`.

**Home flag.** `home_flag = 1` when the driver's nationality matches the
circuit's country (e.g. British driver at Silverstone).

**Era.** F1 rules and car performance shift by technical era. We bucket
years: `v10 and early v8` (2006–2008), `late v8` (2009-2013), `hybrid` (2014–2021), `ground-effect` (2022+).

 **Note on `era`.** `era` will not be included in FEATURES considering the fact that ground-effect era (2022+) never appears 
in training since it is from 2006-2019. We will inspect the efficacy of including `era` on our prediction.

In [ ]:
base_sorted_circuit = (
    base.sort_values(["driverId", "circuitId", "year", "round"])
        .reset_index(drop=True)
)

base_sorted_circuit["circuit_starts_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"]).cumcount()
)
base_sorted_circuit["circuit_podiums_before"] = (
    base_sorted_circuit.groupby(["driverId", "circuitId"])["podium"]
                       .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["circuit_starts_before", "circuit_podiums_before",
                      "circuit_podium_rate",
                      "circuit_starts_before_x", "circuit_starts_before_y",
                      "circuit_podiums_before_x", "circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_sorted_circuit[["raceId", "driverId",
                         "circuit_starts_before",
                         "circuit_podiums_before"]],
    on=["raceId", "driverId"],
    how="left",
)

# Smoothed podium rate.
PRIOR = 0.15
PRIOR_STRENGTH = 5
model["circuit_podium_rate"] = (
    (model["circuit_podiums_before"] + PRIOR * PRIOR_STRENGTH)
    / (model["circuit_starts_before"] + PRIOR_STRENGTH)
)

print(model[["circuit_starts_before", "circuit_podiums_before",
             "circuit_podium_rate"]].describe().round(3))

In [ ]:
first_at_circuit = (
    base_sorted_circuit.sort_values(["driverId", "circuitId", "year", "round"])
                      .groupby(["driverId", "circuitId"])
                      .cumcount() == 0
)
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_starts_before"] == 0).all()
assert (base_sorted_circuit.loc[first_at_circuit, "circuit_podiums_before"] == 0).all()
print("Circuit history leakage check: OK")

In [ ]:
base_team_circuit = (
    base
      .groupby(["constructorId", "circuitId", "raceId", "year", "round"],
               as_index=False)["podium"]
      .mean()
)
base_team_circuit = base_team_circuit.sort_values(
    ["constructorId", "circuitId", "year", "round"]
).reset_index(drop=True)

base_team_circuit["team_circuit_starts_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"]).cumcount()
)
base_team_circuit["team_circuit_podiums_before"] = (
    base_team_circuit.groupby(["constructorId", "circuitId"])["podium"]
                    .transform(lambda s: s.shift(1).fillna(0).cumsum())
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["team_circuit_starts_before",
                      "team_circuit_podiums_before",
                      "team_circuit_starts_before_x",
                      "team_circuit_starts_before_y",
                      "team_circuit_podiums_before_x",
                      "team_circuit_podiums_before_y"]],
    errors="ignore",
)

model = model.merge(
    base_team_circuit[["raceId", "constructorId",
                       "team_circuit_starts_before",
                       "team_circuit_podiums_before"]],
    on=["raceId", "constructorId"],
    how="left",
    validate="many_to_one",
)

print(model[["team_circuit_starts_before",
             "team_circuit_podiums_before"]].describe().round(3))

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "American-Italian": "USA", "Argentine": "Argentina",
    "Argentine-Italian": "Argentina", "Argentinian": "Argentina",
    "Australian": "Australia", "Austrian": "Austria", "Belgian": "Belgium",
    "Brazilian": "Brazil", "British": "UK", "Canadian": "Canada",
    "Chilean": "Chile", "Chinese": "China", "Colombian": "Colombia",
    "Czech": "Czech Republic", "Danish": "Denmark", "Dutch": "Netherlands",
    "East German": "Germany", "Finnish": "Finland", "French": "France",
    "German": "Germany", "Hungarian": "Hungary", "Indian": "India",
    "Indonesian": "Indonesia", "Irish": "Ireland", "Italian": "Italy",
    "Japanese": "Japan", "Liechtensteiner": "Liechtenstein",
    "Malaysian": "Malaysia", "Mexican": "Mexico", "Monegasque": "Monaco",
    "New Zealander": "New Zealand", "Polish": "Poland",
    "Portuguese": "Portugal", "Rhodesian": "Rhodesia", "Russian": "Russia",
    "South African": "South Africa", "Spanish": "Spain", "Swedish": "Sweden",
    "Swiss": "Switzerland", "Thai": "Thailand", "Uruguayan": "Uruguay",
    "Venezuelan": "Venezuela",
}
SECOND_COUNTRY = {"American-Italian": "Italy", "Argentine-Italian": "Italy"}

drivers_nat = tables["drivers"][["driverId", "nationality"]].copy()
drivers_nat["nationality"] = drivers_nat["nationality"].str.strip()

circuits_country = tables["circuits"][["circuitId", "country"]].copy()
circuits_country["country"] = circuits_country["country"].replace(
    "United States", "USA"
)

# Safe to rerun.
model = model.drop(
    columns=[c for c in model.columns
             if c in ["nationality", "country", "home_country",
                      "second_home_country", "home_flag",
                      "nationality_x", "nationality_y",
                      "country_x", "country_y"]],
    errors="ignore",
)

model = model.merge(drivers_nat, on="driverId", how="left", validate="many_to_one")
model = model.merge(circuits_country, on="circuitId", how="left", validate="many_to_one")

model["home_country"] = model["nationality"].map(NATIONALITY_TO_COUNTRY)
model["second_home_country"] = model["nationality"].map(SECOND_COUNTRY)
model["home_flag"] = (
    (model["country"] == model["home_country"]) |
    (model["country"] == model["second_home_country"])
).astype(int)

assert model["nationality"].notna().all()
assert model["country"].notna().all()
unmapped = set(model["nationality"]) - set(NATIONALITY_TO_COUNTRY)
assert len(unmapped) == 0, f"unmapped nationalities: {unmapped}"

# Sanity check: British drivers at Silverstone (circuit 9).
silverstone = model[(model["circuitId"] == 9) & (model["home_flag"] == 1)]
print("UK drivers flagged at Silverstone:", len(silverstone))
print(model["home_flag"].value_counts())

model = model.drop(
    columns=["nationality", "country", "home_country", "second_home_country"],
    errors="ignore",
)

In [ ]:
# Season progress: fraction of the season completed before this race.
season_length = model.groupby("year")["round"].transform("max")
model["season_progress"] = model["round"] / season_length

print(model["season_progress"].describe().round(3))

In [ ]:
model["era"] = pd.cut(
    model["year"],
    bins=[2005, 2008, 2013, 2021, 2024],
    labels=["early_v8", "late_v8", "hybrid", "ground_effect"],
)

# Ordinal encoding so the column is numeric and usable by the pipeline.
# The model treats eras as ordered in time; this is an approximation, see
# the markdown note.
ERA_ORDINAL = {"early_v8": 0, "late_v8": 1, "hybrid": 2, "ground_effect": 3}
model["era_ordinal"] = model["era"].map(ERA_ORDINAL).astype("Int64")

print(model["era"].value_counts())
print("\nera and era_ordinal mapping:")
print(model[["era", "era_ordinal"]].drop_duplicates().sort_values("era_ordinal"))

assert model["era_ordinal"].notna().all(), "era mapping missed a category"
print("\nmodel shape:", model.shape)

In [ ]:
print("Columns in model:")
for i, col in enumerate(sorted(model.columns), 1):
    print(f"{i:2d}. {col}")
print("\nTotal:", len(model.columns))

### Section 3.8: Data dictionary and feature groups

We group the features into six groups for the ablation. Each family group a different kind of pre-race signal:

- **grid**: where the car starts.
- **qualifying**: how fast the driver was in the knockout sessions.
- **standings**: championship position going into the race, for driver
  and constructor.
- **circuit**: the driver's and team's history at this specific track.
- **context**: home race, era, and season progress.

Identifiers (`driverId`, `constructorId`, `circuitId`, `number`) and
keys (`raceId`, `year`, `round`, `date`, `split`) are excluded from
`FEATURES`. They are useful for joins and audit, not as model inputs.

The data dictionary below lists every feature: its group, its source
table, the pre-race fact that justifies it, and its missingness.

In [ ]:
FEATURE_GROUPS = {
    "grid":       ["grid_fixed", "pit_lane_flag"],
    "qualifying": ["qual_position", "best_qual_s", "gap_to_pole_pct",
                   "no_qual_time"],
   "driver_standings": ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before",
                              "cons_standings_missing"],
    "circuit":    ["circuit_starts_before", "circuit_podiums_before",
                   "circuit_podium_rate",
                   "team_circuit_starts_before",
                   "team_circuit_podiums_before"],
    "context":    ["home_flag", "era_ordinal", "season_progress"],
}
FEATURES = [f for group in FEATURE_GROUPS.values() for f in group]

missing = [f for f in FEATURES if f not in model.columns]
assert not missing, f"FEATURES references missing columns: {missing}"

print(f"Total features: {len(FEATURES)}")
for group, cols in FEATURE_GROUPS.items():
    print(f"  {group}: {len(cols)}")

In [ ]:
FORBIDDEN = {"positionOrder", "position", "positionText", "points", "laps",
             "time", "milliseconds", "fastestLap", "rank", "status"}
assert not (set(FEATURES) & FORBIDDEN), \
    f"forbidden in FEATURES: {set(FEATURES) & FORBIDDEN}"
print("All features are leakage-safe.")

In [ ]:
missing = [f for f in FEATURES if f not in model.columns]
assert not missing, f"FEATURES references missing columns: {missing}"
print(f"All {len(FEATURES)} features present in model.")

In [ ]:
import pandas as pd
from IPython.display import display, HTML

# 1. Define your feature groups and source descriptions if you haven't yet
FEATURE_GROUPS = {
    "grid": ["grid_fixed", "pit_lane_flag"],
    "qualifying": ["qual_position", "best_qual_s", "gap_to_pole_pct", "no_qual_time"],
    "driver_standings": ["pts_before", "pos_before", "standings_missing"],
    "constructor_standings": ["cons_pts_before", "cons_pos_before", "cons_standings_missing"],    "circuit": ["circuit_starts_before", "circuit_podiums_before", "circuit_podium_rate", "team_circuit_starts_before", "team_circuit_podiums_before"],
    "context": ["home_flag", "era_ordinal", "season_progress"]
}

FEATURE_SOURCE = {
    "grid_fixed": ("grid", "Starting slot after penalties; pit-lane starts set to PIT_GRID=25."),
    "pit_lane_flag": ("grid", "1 if the driver started from the pit lane."),
    "qual_position": ("qualifying", "Rank in the qualifying session."),
    "best_qual_s": ("qualifying", "Best qualifying lap across the sessions reached."),
    "gap_to_pole_pct": ("qualifying", "Percent slower than the fastest lap in the same race."),
    "no_qual_time": ("qualifying", "1 if the driver set no qualifying time at all."),
    "pts_before": ("driver_standings", "Driver points going into the race."),
    "pos_before": ("driver_standings", "Driver championship position going into the race."),
    "standings_missing": ("driver_standings", "1 if the driver has no previous standings row this season."),
    "cons_pts_before": ("constructor_standings", "Constructor points going into the race."),
    "cons_pos_before": ("constructor_standings", "Constructor championship position going into the race."),
    "cons_standings_missing": ("constructor_standings", "1 if the constructor has no previous standings row this season."),
    "circuit_starts_before": ("derived", "Driver's previous starts at this circuit."),
    "circuit_podiums_before": ("derived", "Driver's previous podiums at this circuit."),
    "circuit_podium_rate": ("derived", "Smoothed podium rate at this circuit."),
    "team_circuit_starts_before": ("derived", "Constructor's previous starts at this circuit."),
    "team_circuit_podiums_before": ("derived", "Constructor's previous podiums at this circuit."),
    "home_flag": ("derived", "1 if the driver's nationality matches the circuit country."),
    "era_ordinal":                         ("derived", "Technical era bucket (early_v8, late_v8, hybrid, ground_effect)."),
    "season_progress": ("derived", "Round divided by season length (0 to 1)."),
}

# 2. Prevent pandas from truncating text columns
pd.set_option('display.max_colwidth', None)

# 3. Loop through and display a separate table for each group
for group, cols in FEATURE_GROUPS.items():
    dict_rows = []
    for f in cols:
        # Safeguard in case a feature isn't in your dataframe yet
        if f in model.columns:
            dtype_val = str(model[f].dtype)
            missing_val = round(model[f].isna().mean(), 4)
        else:
            dtype_val = "N/A"
            missing_val = "N/A"
            
        src, note = FEATURE_SOURCE.get(f, ("unknown", "No description provided"))
        dict_rows.append({
            "feature": f,
            "note": note,
            "missing_share": missing_val,
        })
    
    group_df = pd.DataFrame(dict_rows)
    
    print(f"\nFeature Group: {group.upper()}")
    display(group_df)
    print("-" * 60)

In [ ]:
import json, os

# Build the dictionary from FEATURE_GROUPS and FEATURE_SOURCE.
dict_rows = []
for group, cols in FEATURE_GROUPS.items():
    for f in cols:
        src, note = FEATURE_SOURCE[f]
        dict_rows.append({
            "group": group,
            "feature": f,
            "source": src,
            "missing_share": round(model[f].isna().mean(), 4),
            "note": note,
        })
dictionary = pd.DataFrame(dict_rows)

# Save both artifacts.
os.makedirs("../reports", exist_ok=True)

with open("../reports/p3_feature_groups.json", "w") as f:
    json.dump(FEATURE_GROUPS, f, indent=2)

dictionary.to_csv("../reports/p3_data_dictionary.csv", index=False)

print("saved:")
print(" - reports/p3_feature_groups.json")
print(" - reports/p3_data_dictionary.csv")
print(f" - {len(FEATURES)} features across {len(FEATURE_GROUPS)} groups")

### Section 3.9: Evidence plots

We produce four plots to justify our feature choices:

1. **Podium rate vs grid bucket** shows that starting position predicts podium.
2. **Podium rate vs qualifying bucket** shows that qualifying position predicts podium.
3. **Feature distributions by target** shows standings and circuit features separate podium vs. non-podium.
4. **Correlation heatmap** checks for redundancy and confirms features relate to `podium`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

train = model[model["split"] == "train"].copy()

bins = [0, 2, 4, 6, 10, 15, 24, 25]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16-24", "pit lane (25)"]
train["grid_bucket"] = pd.cut(train["grid_fixed"], bins=bins, labels=labels, right=True)

stats = train.groupby("grid_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats["mean_pct"] = stats["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats.index.astype(str), stats["mean_pct"], color="tab:blue")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats["mean_pct"], stats["count"])], fontsize=8)
ax.set_title("Podium rate by grid position (training years 2006–2019)")
ax.set_xlabel("Grid position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         "Marginal view, not controlled for qualifying or car strength. "
         "Pit-lane starters carry the placeholder grid value 25.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_grid.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats.round(3))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

train = model[model["split"] == "train"].copy()

bins = [0, 2, 4, 6, 10, 15, 25]
labels = ["1-2", "3-4", "5-6", "7-10", "11-15", "16+"]
train["qual_bucket"] = pd.cut(train["qual_position"], bins=bins, labels=labels, right=True)

n_missing = train["qual_position"].isna().sum()
stats_q = train.groupby("qual_bucket", observed=True)["podium"].agg(["mean", "count", "sum"])
stats_q["mean_pct"] = stats_q["mean"] * 100

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(stats_q.index.astype(str), stats_q["mean_pct"], color="tab:orange")
ax.bar_label(bars, labels=[f"{p:.1f}%\n(n={n})" for p, n in
                           zip(stats_q["mean_pct"], stats_q["count"])], fontsize=8)
ax.set_title("Podium rate by qualifying position (training years 2006–2019)")
ax.set_xlabel("Qualifying position")
ax.set_ylabel("Podium rate (%)")
ax.grid(alpha=0.3, axis="y")
ax.set_ylim(0, stats_q["mean_pct"].max() * 1.25)
fig.text(0.5, -0.02,
         f"Marginal view, not controlled for grid. {n_missing} training rows with no qualifying position are excluded.",
         ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("../reports/fig_p3_podium_vs_qual.png", dpi=120, bbox_inches="tight")
plt.show()

print(stats_q.round(3))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

train = model[model["split"] == "train"]

features = ["pts_before", "cons_pts_before", "circuit_podium_rate"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, feat in zip(axes, features):
    x_no = train.loc[train["podium"] == 0, feat].dropna()
    x_yes = train.loc[train["podium"] == 1, feat].dropna()

    # Shared bin edges so the two groups are directly comparable
    edges = np.histogram_bin_edges(pd.concat([x_no, x_yes]), bins=20)

    ax.hist(x_no, bins=edges, density=True, histtype="stepfilled",
            alpha=0.35, color="tab:gray", label=f"not podium (n={len(x_no):,})")
    ax.hist(x_yes, bins=edges, density=True, histtype="step",
            linewidth=3, color="tab:red", label=f"podium (n={len(x_yes):,})")

    ax.set_title(feat)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density (within group)")
    ax.legend(fontsize=8)

fig.suptitle("Distributions by podium outcome, training years 2006–2019 "
             "(densities normalised within each group; podium rate ≈ 14%)",
             fontsize=10)
plt.tight_layout()
plt.savefig("../reports/fig_p3_distributions_by_target.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlations on TRAINING rows only (no validation or test leakage into feature choice)
train = model[model["split"] == "train"]
print("Features in heatmap:", len(FEATURES))
corr = train[FEATURES + ["podium"]].corr()

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink": 0.8}, annot_kws={"size": 7}, ax=ax)
ax.set_title("Feature correlation heatmap (training years 2006–2019 only)")
plt.tight_layout()
plt.savefig("../reports/fig_p3_correlation_heatmap.png", dpi=120, bbox_inches="tight")
plt.show()

print("\nCorrelation with podium (training rows, sorted):")
print(corr["podium"].drop("podium").sort_values(ascending=False).round(3))

### 3.9 findings

All four plots use the training rows only (2006–2019). The correlations are computed on the same split, so they are not influenced by validation or test.

**Plot 1 (grid).** Podium rate falls with each step down the grid:

| Grid bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 72.9% | 535 |
| 3-4 | 43.7% | 535 |
| 5-6 | 17.5% | 536 |
| 7-10 | 4.9% | 1,068 |
| 11-15 | 1.4% | 1,331 |
| 16-24 | 0.9% | 1,717 |
| pit lane (25) | 0.0% | 20 |

The pit-lane group shows no podium finishes, but 20 starts are too few to estimate a rate. Pit-lane starts are therefore represented by `pit_lane_flag`, and `grid_fixed = 25` places them behind every real grid slot.

**Plot 2 (qualifying).** The same shape:

| Qualifying bucket | Podium rate | Count |
|---|---:|---:|
| 1-2 | 71.8% | 536 |
| 3-4 | 43.2% | 535 |
| 5-6 | 20.0% | 535 |
| 7-10 | 4.3% | 1,067 |
| 11-15 | 1.6% | 1,331 |
| 16+ | 0.8% | 1,718 |

Grid and qualifying position are closely related because most drivers start where they qualify. They are not identical: pit-lane starts have a qualifying position but no grid slot, and grid penalties move some drivers back from their qualifying position. 

**Plot 3 (distributions by target).**

- `pts_before`: podium drivers have more pre-race points. The podium
  group has a long right tail; the non-podium group is concentrated near
  zero. Clear separation.
- `cons_pts_before`: the same pattern, slightly stronger. The podium
  distribution is shifted further right than the driver version,
  consistent with car performance being the dominant factor.
- `circuit_podium_rate`: the podium group is shifted slightly right, but
  the overlap is large. The feature has signal but is weaker than the
  standings.

**Plot 4 (correlations with podium, training rows only).**

The ranking matches the expected structure. Grid and qualifying are the
strongest features (|r| ≈ 0.50). Driver and constructor standings follow
(|r| ≈ 0.44–0.47). Circuit features are moderate (0.10–0.29). The flags
and `season_progress` are near zero.

- `circuit_podium_rate` (0.285) is stronger than the raw counts
  (`circuit_podiums_before` 0.250, `circuit_starts_before` 0.168),
  supporting the smoothed rate over the counts.
- `best_qual_s` (-0.057) is weak because absolute lap times vary by
  track; `gap_to_pole_pct` (-0.290) is the normalised version and carries
  five times the signal.
- `season_progress` is exactly zero on its own. That is expected: it is
  not meant to predict podium directly, but to modulate `pts_before`.
  Whether the model can use it is a question for the ablation.
  `era_ordinal` is 0.012, essentially zero. Era is a time bucket, not a
per-race property, so a linear correlation cannot capture it. The
ablation is the right test.
- The rare-event flags (`no_qual_time`, `pit_lane_flag`,
  `*_missing`) have small correlations because they fire on few rows.
  Correlation is the wrong measure for them; the ablation is the right
  one.

No feature pair exceeds 0.95 correlation, so no two features are
redundant enough to drop on correlation alone. `grid_fixed` and
`qual_position` are the highest pair at ~0.9, and both stay: they carry
distinct information and the ablation shows the grid group contributes
the largest drop.

### Section 3.10: Preprocessing experiments

We run four experiments, each with the same logistic regression as the
baseline. Only the preprocessing step changes.

- **1a: scaling vs no scaling.** Does `StandardScaler` change performance
  or convergence?
- **1b: order (impute→scale vs scale→impute).** Does the order matter?
- **2: median vs mean imputation.** Which fill value is better for this
  feature set?
- **3: fit-on-train vs fit-on-all (leakage demo).** Does fitting the
  preprocessor on validation rows inflate the score?

All experiments use the same train/validation split and the same
`LogisticRegression(max_iter=2000, random_state=SEED)`.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.compose import ColumnTransformer

# Ensure splits are defined 
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

X_tr = train_df[FEATURES];  y_tr = train_df["podium"].values
X_va = val_df[FEATURES];    y_va = val_df["podium"].values

print("X_tr:", X_tr.shape, " X_va:", X_va.shape)
print("\n")

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

def lr_pipeline(scale: bool):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("lr", LogisticRegression(max_iter=2000, random_state=SEED)))
    return Pipeline(steps)

def run_and_count_warnings(pipe, X_fit, y_fit, X_eval, y_eval):
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        pipe.fit(X_fit, y_fit)
        proba = pipe.predict_proba(X_eval)[:, 1]
    n_warn = sum(1 for x in w if issubclass(x.category, ConvergenceWarning))
    return {
        "PR-AUC": round(average_precision_score(y_eval, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_eval, proba), 4),
        "warnings": n_warn,
    }

import pandas as pd
rows = []
for name, scale in [("impute→scale", True), ("impute-only", False)]:
    r = run_and_count_warnings(lr_pipeline(scale=scale), X_tr, y_tr, X_va, y_va)
    rows.append({"pipeline": name, **r})

exp1a = pd.DataFrame(rows)
print("Experiment 1a: scaling vs no scaling (validation)")
print(exp1a.to_string(index=False))

In [ ]:
pipe_imp_sc = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc",  StandardScaler()),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])
pipe_sc_imp = Pipeline([
    ("sc",  StandardScaler()),
    ("imp", SimpleImputer(strategy="median")),
    ("lr",  LogisticRegression(max_iter=2000, random_state=SEED)),
])

print("Experiment 1b: order of imputation and scaling")
for name, pipe in [("impute→scale", pipe_imp_sc), ("scale→impute", pipe_sc_imp)]:
    pipe.fit(X_tr, y_tr)
    pr = average_precision_score(y_va, pipe.predict_proba(X_va)[:, 1])
    print(f"  {name}: validation PR-AUC = {pr:.4f}")

In [ ]:
def lr_pipeline_imputer(strategy: str):
    return Pipeline([
        ("imputer", SimpleImputer(strategy=strategy)),
        ("scaler",  StandardScaler()),
        ("lr",      LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

rows = []
for name, strategy in [("median", "median"), ("mean", "mean")]:
    pipe = lr_pipeline_imputer(strategy)
    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_va)[:, 1]
    rows.append({
        "imputer": name,
        "PR-AUC":  round(average_precision_score(y_va, proba), 4),
        "ROC-AUC": round(roc_auc_score(y_va, proba), 4),
    })

exp1c = pd.DataFrame(rows)
print("Experiment 1c: median vs mean imputation (validation)")
print(exp1c.to_string(index=False))

In [ ]:
median_imp = SimpleImputer(strategy="median").fit(X_tr)
mean_imp   = SimpleImputer(strategy="mean").fit(X_tr)

compare = pd.DataFrame({
    "feature":       FEATURES,
    "median_fill":   median_imp.statistics_.round(3),
    "mean_fill":     mean_imp.statistics_.round(3),
    "diff":          (mean_imp.statistics_ - median_imp.statistics_).round(3),
    "missing_share": [round(X_tr[f].isna().mean(), 4) for f in FEATURES],
})
print(compare[compare["missing_share"] > 0].to_string(index=False))

In [ ]:
from sklearn.base import clone

prep_template = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

def lr_on(prep, X, y):
    return LogisticRegression(max_iter=2000, random_state=SEED).fit(prep.transform(X), y)

prep_correct = clone(prep_template).fit(X_tr)
clf_correct  = lr_on(prep_correct, X_tr, y_tr)
proba_correct = clf_correct.predict_proba(prep_correct.transform(X_va))[:, 1]

X_tr_va = pd.concat([X_tr, X_va])
prep_leaky = clone(prep_template).fit(X_tr_va)
clf_leaky  = lr_on(prep_leaky, X_tr, y_tr)
proba_leaky = clf_leaky.predict_proba(prep_leaky.transform(X_va))[:, 1]

pr_correct = average_precision_score(y_va, proba_correct)
pr_leaky   = average_precision_score(y_va, proba_leaky)

print("Experiment 2: preprocessing fit on train vs train+val")
print(f"  Correct (train only) : PR-AUC = {pr_correct:.4f}")
print(f"  Leaky   (train+val)  : PR-AUC = {pr_leaky:.4f}")
print(f"  Difference           : {pr_leaky - pr_correct:+.4f}")

scaler_correct = prep_correct.named_steps["scaler"]
scaler_leaky   = prep_leaky.named_steps["scaler"]
mean_shift_std  = np.abs(scaler_correct.mean_ - scaler_leaky.mean_) / scaler_correct.scale_
scale_shift_pct = np.abs(scaler_correct.scale_ - scaler_leaky.scale_) / scaler_correct.scale_ * 100

print(f"\nScaler drift: mean shift {mean_shift_std.mean():.4f} std (max {mean_shift_std.max():.4f})")
print(f"              scale change {scale_shift_pct.mean():.2f}% (max {scale_shift_pct.max():.2f}%)")

In [ ]:
for strategy in ["median", "mean"]:
    pipe = lr_pipeline_imputer(strategy)
    pipe.fit(X_tr, y_tr)
    fills = pipe.named_steps["imputer"].statistics_
    print(f"\n{strategy} fills:")
    for f, v in zip(FEATURES, fills):
        if X_tr[f].isna().any():
            print(f"  {f}: {v:.3f}")

    proba = pipe.predict_proba(X_va)[:, 1]
    print(f"  PR-AUC: {average_precision_score(y_va, proba):.4f}")

### 3.10 findings

**Experiment 1a: scaling vs no scaling.**

| Pipeline | PR-AUC | ROC-AUC | Convergence warnings |
|---|---:|---:|---:|
| impute→scale | 0.7353 | 0.9297 | 0 |
| impute-only  | (invalid) | (invalid) | 1 |

The unscaled pipeline fails to converge: lbfgs hits the 2000-iteration
limit. Its PR-AUC is not a valid comparison because the model did not
finish optimizing. The conclusion is not "scaling helps accuracy" but
"scaling is required for the optimizer to converge on this feature
set". We use `impute→scale` for that reason.

**Experiment 1b: order of imputation and scaling.**

| Order | Validation PR-AUC |
|---|---:|
| impute→scale | 0.7353 |
| scale→impute | 0.7354 |

Identical to 3 decimals. We use `impute→scale`, the standard
scikit-learn order, which produces a final distribution with mean 0 and
std 1.

**Experiment 2: median vs mean imputation (validation).**

We compared `SimpleImputer(strategy="median")` and
`SimpleImputer(strategy="mean")` with the same downstream pipeline. The
fill values differ materially on the skewed features:

| Feature | Median fill | Mean fill | Difference |
|---|---:|---:|---:|
| `gap_to_pole_pct` | 1.828 | 2.376 | +0.547 |
| `pos_before` | 11.000 | 11.372 | +0.372 |
| `qual_position` | 11.000 | 11.270 | +0.270 |
| `best_qual_s` | 89.103 | 89.274 | +0.171 |
| `cons_pos_before` | 6.000 | 5.896 | -0.104 |
| `standings_missing` | 0.000 | 0.058 | +0.058 |
| `cons_standings_missing` | 0.000 | 0.052 | +0.052 |

But validation PR-AUC is identical to 4 decimals for both (0.7353 /
0.7353). The reason is that the features with the largest fill difference
also have the smallest missingness: `gap_to_pole_pct` is missing in 1.3%
of training rows, so the fill affects ~74 of 5,742 rows. Changing those
values does not change the model's ranking enough to move the metric.

We choose **median**. It is the standard choice for skewed features,
and the fill value for `gap_to_pole_pct` (right-skewed, max 45.6%) is
0.55 percentage points lower than the mean, which better represents the
typical non-missing value. The two are equivalent on this validation
set, so the choice is about robustness, not accuracy.

**Experiment 3: fit on train vs train+val (leakage demo).**

| Setup | Validation PR-AUC |
|---|---:|
| Fit on train only (correct) | 0.7353 |
| Fit on train+val (leaky) | 0.7355 |
| Difference | +0.0002 |

The metric impact is small (imputation and scaling are mild
transformations), but the scaler's parameters drift measurably:
per-feature mean shift 0.0137 std on average (max 0.0591), scale change
2.46% on average (max 25.81% on one feature). The leak is real. We fit
on train only because it is the honest rule and the effect could be
larger under a bigger distribution shift.

### Section 3.11: Feature-group ablation

We measure each feature group's contribution by dropping one group at a time, 
retraining the same model, and recording the drop in validation PR-AUC.

**Setup:** same model as the baseline (logistic regression), same 
preprocessing (impute median + scale), same train/val split. Only the feature 
set changes.

**Uses `train_eval()`** (from `src/f1_eval.py`) so the metrics are 
directly comparable to the baseline model's comparison table.

**Reading the results:** a large drop means the group is important. A near-zero 
or negative drop means the group adds nothing (or its signal is redundant with 
other groups).

In [ ]:
import sys
sys.path.append("../src")
from f1_eval import train_eval

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Ensure splits and targets are defined.
train_df = model[model["split"] == "train"]
val_df   = model[model["split"] == "validation"]
test_df  = model[model["split"] == "test"]

y_train = train_df["podium"].values
y_val   = val_df["podium"].values
y_test  = test_df["podium"].values

def make_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, random_state=SEED)),
    ])

print(f"Features: {len(FEATURES)}  Groups: {len(FEATURE_GROUPS)}")
print(f"Train: {train_df.shape}  Val: {val_df.shape}")

In [ ]:
results = {}

# Baseline: all features
df = train_eval(
    make_pipeline(),
    train_df[FEATURES], y_train,
    val_df[FEATURES],   y_val,
    name="baseline (all features)",
)
baseline_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
results["baseline"] = baseline_pr

print(f"Baseline validation PR-AUC (all features): {baseline_pr:.4f}")
print(df.round(4))

In [ ]:
results = {"baseline": baseline_pr}

for group, cols in FEATURE_GROUPS.items():
    dropped_features = [f for f in FEATURES if f not in cols]

    df = train_eval(
        make_pipeline(),
        train_df[dropped_features], y_train,
        val_df[dropped_features],   y_val,
        name=f"without {group}",
    )
    val_pr = df[df["split"] == "validation"]["PR-AUC"].iloc[0]
    results[f"without {group}"] = val_pr
    print(f"without {group:12s} (dropped {len(cols)} features): PR-AUC = {val_pr:.4f}  drop = {baseline_pr - val_pr:+.4f}")

In [ ]:
import pandas as pd

rows = []
for name, pr in results.items():
    if name == "baseline":
        continue
    rows.append({
        "dropped group": name.replace("without ", ""),
        "PR-AUC": round(pr, 4),
        "drop vs baseline": round(baseline_pr - pr, 4),
    })

ablation_table = pd.DataFrame(rows).sort_values("drop vs baseline", ascending=False)
print(f"Baseline PR-AUC: {baseline_pr:.4f}")
print()
print(ablation_table.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
sorted_table = ablation_table.sort_values("drop vs baseline")
values = sorted_table["drop vs baseline"]
labels = sorted_table["dropped group"]

bars = ax.barh(labels, values, color="tab:blue")

# Place labels with proper alignment based on sign
for bar, val in zip(bars, values):
    if val >= 0:
        ax.text(val + 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="left", fontsize=9)
    else:
        ax.text(val - 0.0004, bar.get_y() + bar.get_height()/2,
                f"{val:+.4f}", va="center", ha="right", fontsize=9)

ax.axvline(0, color="black", linestyle="-", linewidth=0.8)

# Give some padding so labels don't touch the plot edges
x_min, x_max = values.min(), values.max()
padding = (x_max - x_min) * 0.15
ax.set_xlim(x_min - padding, x_max + padding)

ax.set_title("Feature-group ablation: validation PR-AUC drop when group is removed")
ax.set_xlabel("PR-AUC drop vs baseline (larger = more important)")
ax.set_ylabel("")
ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.savefig("../reports/fig_p3_ablation.png", dpi=120, bbox_inches="tight")
plt.show()

### 3.11 findings

Baseline validation PR-AUC (20 features): 0.7364.

| Dropped group | PR-AUC | Drop vs baseline |
|---|---:|---:|
| driver_standings | 0.7017 | +0.0347 |
| grid | 0.7280 | +0.0084 |
| qualifying | 0.7288 | +0.0076 |
| context | 0.7322 | +0.0042 |
| constructor_standings | 0.7354 | +0.0010 |
| circuit | 0.7374 | **−0.0010** |

**Interpretation.**

- **Driver standings is the strongest group.** Dropping `pts_before`,
  `pos_before`, and `standings_missing` costs 0.0347 PR-AUC, more than
  any other group. The driver's championship position going into the
  race is the single most informative pre-race signal.
- **Grid is second.** Dropping `grid_fixed` and `pit_lane_flag` costs
  0.0084.
- **Qualifying is third.** Dropping the qualifying group costs 0.0076.
- **Context contributes +0.0042.** `home_flag`, `era_ordinal`, and
  `season_progress` together. Adding `era_ordinal` raised the group's
  drop from +0.0031 to +0.0042 and the baseline from 0.7353 to 0.7364.
- **Constructor standings is nearly redundant.** +0.0010 is within noise.
- **Circuit is negative.** Dropping the entire circuit group *improves*
  validation PR-AUC by 0.0010. We drop it from `FEATURES`.

**Decision.** Remove the circuit group. The remaining five groups all
contribute positively.